# BLIP-2 ITM re-ranking — pipeline stage 4

**What this notebook shows.** Online step 4 of the pipeline: BLIP-2 image-text matching is computed for each candidate caption against the query image, and candidates are re-ranked by the ITM probability.

**Model.** `Salesforce/blip2-itm-vit-g` — BLIP-2 with the image-text matching head. ~5 GB VRAM in fp16.

**Headline finding (see `report.md` §6.2 #5).** ITM re-ranking is a *negative ablation* on this dataset:
- Pure-ITM reordering of the top-50 ANN candidates **hurts** NDCG@10 by 3 pp and mAP@10 by 3 pp.
- A low-weight blend (`combined = 0.8·ANN + 0.2·ITM`) recovers to baseline but adds no value.
- Reason: BLIP-2 captions for DeepFashion are short product descriptors ("black floral print mini dress") with limited vocabulary. The top-K visually-similar candidates share near-identical captions, so the ITM score can't discriminate among them.

This notebook reproduces the negative finding on a small sample so the mechanism is visible.

In [ ]:
# ── 1. Setup ──────────────────────────────────────────────────────────────
!pip install -q transformers accelerate

In [ ]:
import json
from pathlib import Path

import torch
from PIL import Image
from transformers import AutoProcessor, Blip2ForImageTextRetrieval

if Path('/kaggle/input').exists():
    DATASET_ROOT  = Path('/kaggle/input/datasets/ashok1145/vr-fproj/vr_final_proj_dataset')
    CAPTIONS_FILE = Path('/kaggle/input/datasets/taralsanka/blip-updated-captions/captions.json')
else:
    DATASET_ROOT  = Path('vr_final_proj_dataset')
    CAPTIONS_FILE = Path('artifacts/captions.json')

IMG_ROOT  = DATASET_ROOT / 'img' / 'img'
BBOX_FILE = DATASET_ROOT / 'Anno' / 'list_bbox_inshop.txt'
SPLIT_FILE = DATASET_ROOT / 'eval' / 'list_eval_partition.txt'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
print('CAPTIONS_FILE exists:', CAPTIONS_FILE.exists())

In [ ]:
# ── 2. Load BLIP-2 ITM (≈ 5 GB in fp16) ───────────────────────────────────
MODEL = 'Salesforce/blip2-itm-vit-g'
processor = AutoProcessor.from_pretrained(MODEL)
dtype = torch.float16 if DEVICE == 'cuda' else torch.float32
model = Blip2ForImageTextRetrieval.from_pretrained(MODEL, torch_dtype=dtype).to(DEVICE).eval()
print('BLIP-2 ITM ready.')

In [ ]:
# ── 3. Scoring helper ─────────────────────────────────────────────────────
@torch.no_grad()
def itm_probs(pil_image, captions_list):
    """Return the 'match' probability for each (image, caption) pair."""
    inputs = processor(images=[pil_image]*len(captions_list), text=captions_list,
                       return_tensors='pt', padding=True).to(DEVICE)
    if DEVICE == 'cuda':
        inputs = {k: (v.half() if v.dtype.is_floating_point else v) for k, v in inputs.items()}
    out = model(**inputs, use_image_text_matching_head=True)
    # Output shape: (B, 2). Softmax over the 'no match' / 'match' columns.
    probs = torch.softmax(out.logits_per_image.float(), dim=-1)[:, 1]
    return probs.cpu().tolist()

In [ ]:
# ── 4. Pick a query image + 5 candidate captions ──────────────────────────
# We pretend the user uploaded a query image. Find 5 candidate captions of
# similar-looking products in the gallery — exactly what ANN search would return.

def parse_split(p):
    with open(p) as f: lines = f.readlines()
    rows = []
    for line in lines[2:]:
        parts = line.strip().split()
        if len(parts) < 3: continue
        raw = parts[0]
        nm  = raw[len('img/'):] if raw.startswith('img/') else raw
        rows.append({'image_name': nm, 'item_id': parts[1], 'split': parts[2]})
    return rows

rows = parse_split(SPLIT_FILE)
captions = json.load(open(CAPTIONS_FILE))

# Query: a 'query' split image
q = next(r for r in rows if r['split'] == 'query')
query_pil = Image.open(IMG_ROOT / q['image_name']).convert('RGB')

# Candidates: 5 random gallery items (in real pipeline these come from HNSW top-K)
import random; random.seed(0)
gallery = [r for r in rows if r['split'] == 'gallery' and r['image_name'] in captions]
cands = random.sample(gallery, 5)
cand_caps = [captions[r['image_name']] for r in cands]
for r, c in zip(cands, cand_caps):
    print(f'  {r["item_id"]:<18} "{c}"')

In [ ]:
# ── 5. ITM scores ─────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

scores = itm_probs(query_pil, cand_caps)

fig, axes = plt.subplots(1, 6, figsize=(22, 5))
axes[0].imshow(query_pil); axes[0].axis('off'); axes[0].set_title(f'QUERY\n{q["item_id"]}', fontsize=10)
for ax, r, cap, sc in zip(axes[1:], cands, cand_caps, scores):
    cand_pil = Image.open(IMG_ROOT / r['image_name']).convert('RGB')
    ax.imshow(cand_pil); ax.axis('off')
    ax.set_title(f'{r["item_id"]}\nITM={sc:.3f}\n"{cap}"', fontsize=9)
plt.tight_layout(); plt.show()

print(f'\nITM probability range across these 5 candidates: '
      f'{min(scores):.3f} – {max(scores):.3f}  '
      f'(spread = {max(scores)-min(scores):.3f})')

### Why ITM doesn't help on this dataset

Look at the ITM scores above. On any 5 visually-similar (or even random) candidates with short product captions, the ITM probability **clusters in a narrow range** — typically 0.6 ± 0.1 across all candidates, or sometimes all 0.95 ± 0.02. This is because:

1. BLIP-2's ITM head was trained on COCO scene captions ("a person riding a horse on the beach"), where individual sentences are far more discriminative.
2. Our captions are short product tags ("black floral print mini dress"). Many gallery items share near-identical captions, so the ITM score can't distinguish *which* black floral dress is the right one for this specific query.
3. The strong signal — visual similarity — is already encoded in the ANN cosine score. Re-ranking by ITM throws that signal away in favour of a coarse text-match signal.

**Full-set numbers from `report.md` §6**:

| Pipeline | R@10 | NDCG@10 | mAP@10 |
| --- | --- | --- | --- |
| C-HN α=0.7 (no rerank) | 0.881 | 0.578 | 0.480 |
| C-HN α=0.7 + ITM blend(0.2) | 0.881 | 0.551 | 0.449 |

R@10 is unchanged, but the ranking metrics drop ~3 pp. We report this as a **negative ablation**: the architecture from the problem statement includes ITM re-ranking, we implemented it faithfully, and we measured a real (small) regression.